# Tutorial 02: Multi-Worker Empirical Benchmarking & Hugging Face TrackIO Logging

In this notebook, we demonstrate running multi-worker parallel benchmark evaluation across **Memory Replay**, **LoRA Online**, and **Hybrid Baselines** connected to your local **LM Studio** server endpoint, and logging metrics to **Hugging Face TrackIO**.

## 1. Import ExperimentTracker & Parallel Benchmark Suite

In [ ]:
import os
from open_continual_env.benchmark import ParallelBenchmarkRunner, ExperimentTracker, ContinualMetrics

# Initialize Hugging Face TrackIO Experiment Tracker
tracker = ExperimentTracker(project_name="open_continual_env", experiment_name="notebook_demo")
print(f"TrackIO Enabled: {tracker.use_trackio}")

## 2. Execute Multi-Worker Benchmark Run

Run benchmark across 4 parallel workers on LM Studio endpoint `http://127.0.0.1:1234/v1`.

In [ ]:
from benchmarks.run_empirical_benchmark import run_empirical_benchmark

# Execute multi-worker empirical benchmark evaluation
results = run_empirical_benchmark(
    api_base="http://127.0.0.1:1234/v1",
    model_name="google/gemma-4-e4b",
    num_episodes=3,
    output_dir="benchmark_results"
)

print("Benchmark Completed! Summarized Baselines:")
for b_name, b_metrics in results.get("baselines", {}).items():
    print(f" - {b_name}: Pass Rate = {b_metrics['task_success_rate']*100:.1f}%, BWT = {b_metrics['backward_transfer']:.3f}")

## 3. Generate Performance & Forgetting Trajectory Plots

Using `ExperimentTracker.generate_plots()` to visualize learning curves.

In [ ]:
# Log summary metrics to TrackIO
for b_name, b_metrics in results.get("baselines", {}).items():
    tracker.log({
        f"{b_name}/success_rate": b_metrics["task_success_rate"],
        f"{b_name}/reward": b_metrics["mean_reward"],
        f"{b_name}/backward_transfer": b_metrics["backward_transfer"],
    })

plot_paths = tracker.generate_plots(output_dir="benchmark_results/plots")
print(f"Generated Trajectory Plots: {plot_paths}")